# Chapter 16 — Testing the Capstone Agent

*Coverage, judgment, attribution — applied to the real Chapter 15 complaint agent.*

The 20-case benchmark tells you the agent handles the twenty situations you thought of. This chapter puts the finished agent on a test stand: a factor-balanced suite of complaint scenarios, the actual governed agent under each, a grounded judgment of its drafts, and an attribution of failures to specific input factors.

The heavy 120-scenario run is precomputed by `scripts/run_capstone_doe.py` into `data/capstone_doe_results.csv` (deterministic, greedy decoding). This notebook reproduces the design, the judgment method, and the attribution from that CSV — without re-running the agent.

In [ ]:
from pathlib import Path
import pandas as pd
# CapstoneTestHarness ships in the licensed *Beyond Ship and Pray, Pro Edition*
# (knowlytix). The open proofloop package includes the rest of the testing tools.
try:
    from proofloop.testing import CapstoneTestHarness, CapstoneTestResult
except ImportError:
    CapstoneTestHarness = CapstoneTestResult = None
    print('CapstoneTestHarness requires the licensed Pro edition — see https://knowlytix.ai/')

stand = CapstoneTestHarness(n_runs=120, seed=42)
print('harness ready')

## Stage 1+2 — design the complaint suite

Three presentation factors (clarity, entity aliasing, reasoning cue) crossed with the 20 seed cases as a blocking factor. The Sobol design balances every level and covers every level-pair.

In [ ]:
design = stand.design()
print(f'{len(design)} scenarios')
for col in ('clarity', 'entity_aliasing', 'reasoning_cue'):
    print(f'  {col:<16} {design[col].value_counts().to_dict()}')
print(f'  seed_case        {design["seed_case"].nunique()} cases, '
      f'{design["seed_case"].value_counts().min()}-{design["seed_case"].value_counts().max()} each')

Each design row is materialized into a concrete message by deterministic templated transforms — the dollar amount (the verifiable claim) survives, everything around it varies.

In [ ]:
seed = {'seed_case': 'case-001', 'clarity': 'ambiguous',
        'entity_aliasing': 'typo', 'reasoning_cue': 'none'}
print(stand.materialize(seed))

## Stage 3 — run the governed agent (precomputed)

Each scenario runs through `build_complaint_harness()` exactly as the capstone built it. We load the precomputed results rather than re-running 120 agent trajectories here.

In [ ]:
CSV = Path('../data/capstone_doe_results.csv')
if not CSV.exists():
    CSV = Path('data/capstone_doe_results.csv')
df = pd.read_csv(CSV, keep_default_na=False, na_values=[''])
print(f'{len(df)} scenarios')
print(f'  overall accuracy: {df["correct"].mean():.3f}')
print(f'  status counts:    {df["status"].value_counts().to_dict()}')

Because this is a fixed-workflow agent, the test scores the **whole trajectory**, not just the final label: the outcome (classification + escalation vs labels), the trajectory structure (did the workflow run in order, and did it escalate by the right path), and process health (steps, tool failures, clean finish).

In [ ]:
def _rate(col):
    s = df[col]
    return (s.astype(str) == 'True').mean() if s.dtype == object else s.mean()

print(f'  outcome accuracy:    {df["correct"].mean():.3f}')
print(f'  trajectory accuracy: {df["trajectory_ok"].mean():.3f}')
print(f'  workflow adherence:  {_rate("workflow_adherent"):.3f}')
print(f'  escalation-path acc: {_rate("trigger_correct"):.3f}')

Reducing each run to one bit would hide *how* it reached its verdict. The executed tool sequence and escalation path show it directly — a PII case refused at the first tool call, a UDAAP case escalating only after `flag_regulatory`.

In [ ]:
for cid in ['case-011', 'case-016', 'case-002']:   # a PII, a UDAAP, a failed run
    r = df[df['seed_case'] == cid].iloc[0]
    print(f"{cid}  status={r['status']}")
    print(f"   tools:   {r['tool_sequence']}")
    print(f"   trigger: {r['escalation_trigger']}  (expected {r['expected_trigger']})")

## Stage 4 — judgment: correctness and groundedness

Correctness compares classification + escalation to the case labels. Groundedness scores a draft's numeric claim as distance from the trained substrate (`score_triple`), with per-relation calibrated bands. A committed value is *grounded*; a wrong or unknown one is *fabrication*.

In [ ]:
for amount in (35, 45, 50):
    triple, geo, tier = stand.judge_draft(
        f'The overdraft fee is ${amount} per occurrence.', 'overdraft_fee')
    print(f'  ${amount}: {tier:<12} geodesic={geo:.3f}')

print('\nsuite tier distribution:', df['draft_tier'].value_counts().to_dict())

On a free-text draft with no extracted claim the full geometric judge collapses to a constant geodesic and labels everything fabrication — which is why the stand aligns the one verifiable claim itself. Across the suite the check is narrow: most scenarios escalate or cite deadlines the store does not hold as triples, so they score `n/a`; the fifteen drafts stating a checkable amount (the \$35 overdraft fee at 0.865, the \$35 reversal cap at 0.939) are each confirmed grounded.

## Stage 5 — attribution

Logistic regression of `correct` on the three presentation factors (the seed case is a blocking factor, excluded), with Benjamini–Hochberg correction.

In [ ]:
attribution = stand.analyze(CapstoneTestResult(rows=df.to_dict('records')))

for row in attribution.logistic_table:
    flag = '*' if row.get('significant_adj') else ' '
    print(f' {flag} {row["factor"]:<16} p_adj={row["p_value_adj"]:.4f}  '
          f'pseudo_r2={row["pseudo_r2"]:.3f}')

print()
for row in attribution.top_failures(k=3):
    print(f'   {row["factor"]:<16} {row["level"]:<14} '
          f'failure_rate={row["failure_rate"]:.2f}')

`clarity` is the factor that survives correction: its `misleading` level — understating an escalation-worthy complaint — drives the failures. The agent is most fragile to a customer who downplays the problem, exactly the input a complaint pipeline most needs to get right.

## Which component failed: per-tool decomposition

Factor attribution names the *input* that breaks the agent; per-tool decomposition names the *component*. Each tool's output (in the CSV) is scored against the case's per-tool ground truth, and each end-to-end failure is blamed on the first tool, in workflow order, that erred.

In [ ]:
import json
p = Path('../data/capstone_tool_breakdown.json')
if not p.exists(): p = Path('data/capstone_tool_breakdown.json')
obj = json.loads(p.read_text())
for tool, s in obj['per_tool'].items():
    acc = f"{s['accuracy']:.2f}" if s['accuracy'] is not None else 'n/a'
    print(f'  {tool:<18} accuracy={acc}  ({s["correct"]}/{s["scored"]})')
print('  weak-link blame:', obj['weak_link_counts'])

## Resilience: faulting each tool (real knowlytix ToolGateway)

The `knowlytix` `ToolGateway` (Ch12 of the testing monograph) installs on the capstone's executor and injects a fault into one tool at a time. A governed agent must **fail loud** on a broken tool, never silently pass a corrupted result downstream. (`reached < n`: some cases escalate before reaching that tool.)

In [ ]:
import json
p = Path('../data/capstone_fault_injection.json')
if not p.exists(): p = Path('data/capstone_fault_injection.json')
obj = json.loads(p.read_text())
for tool, s in obj.items():
    rate = f"{s['detection_rate']:.2f}" if s['detection_rate'] is not None else 'n/a'
    print(f'  {tool:<18} detected={s["detected"]}/{s["reached"]} reached  '
          f'silent={s["silent"]}  detection_rate={rate}')

## Testing the substrate, and the ship/no-ship gate

The full `knowlytix` platform in **native mode** (`DOEGMSBenchmark`): ingest the policy doc, generate geometric questions, run a Qwen-RAG knowledge evaluator through the *calibrated* verifier pipeline, and end on a risk-tiered release gate. This tests the ground truth the agent stands on.

In [ ]:
import json
p = Path('../data/capstone_substrate_test.json')
if not p.exists(): p = Path('data/capstone_substrate_test.json')
obj = json.loads(p.read_text())
print(f"  ground-truth baseline: {obj['baseline_accuracy']}")
print(f"  Qwen-RAG substrate:    {obj['evaluator_accuracy']}  ({obj['n_questions']} questions)")
for g in obj['gates']:
    verdict = 'SHIP' if g['passed'] else 'NO-SHIP'
    print(f"  {g['tier']:<14} {verdict}  (acc {g['actual']:.2f} vs threshold {g['threshold']})")

## Testing the retriever against a policy-derived ground truth

The per-tool `0.42` was retrieval *recall*; the substrate test scored a stand-in reader. This points the same native machinery at the **real** `search_policy`: build a GMS from the policy (the answer key), reverse it into customer-style queries, run retrieve-*then*-generate, decompose each generated answer into typed claims, and verify each claim against the GMS. `claims_verified / n_claims` is a **joint** retrieval-and-generation faithfulness score (a wrong retrieval yields claims that fail verification); the failure codes say where the rest broke. Precomputed by `stand.rag_test()` (~3.5 min) into `data/capstone_rag_test.json`.

In [ ]:
import json
p = Path('../data/capstone_rag_test.json')
if not p.exists(): p = Path('data/capstone_rag_test.json')
obj = json.loads(p.read_text())
print(f"  questions={obj['n_questions']}  "
      f"claims verified={obj['claims_verified']}/{obj['n_claims']}  "
      f"mean conf={obj['mean_confidence']}")
print(f"  verifier operating point (geodesic) = {obj['plausibility_threshold']:.2f}")
for code, n in sorted(obj['failure_codes'].items(), key=lambda kv: -kv[1]):
    print(f"    {code:<18} {n}")

## Exercise (worked) — watch a tier flip

Edit a grounded draft's figure to a wrong value and confirm the tier flips; restore it and confirm it returns. This is the byte-level sensitivity the substrate buys.

In [ ]:
good = 'Per fee reversal policy, a rep may approve up to $35; above needs manager approval.'
bad  = good.replace('$35', '$80')
for label, text in [('committed', good), ('drifted', bad)]:
    triple, geo, tier = stand.judge_draft(text, 'general')
    print(f'  {label:<10} {triple}  geodesic={geo if geo is None else round(geo,3)}  -> {tier}')

In [ ]:
# Self-check: design is balanced, CSV loaded, attribution non-empty, tiers separate.
assert len(design) == 120
assert {'clarity','entity_aliasing','reasoning_cue'} <= {r['factor'] for r in attribution.logistic_table}
assert stand.judge_draft('The overdraft fee is $35 per occurrence.', 'overdraft_fee')[2] == 'grounded'
print('self-check OK')